In [1]:
# unable colab to access git
!git clone https://github.com/yifeica0/ECS171-Group8.git
%cd ECS171-Group8
!git checkout models

fatal: destination path 'ECS171-Group8' already exists and is not an empty directory.
/content/ECS171-Group8
Already on 'models'
Your branch is up to date with 'origin/models'.


In [12]:
# update repo informations
!git pull

Already up to date.


In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, roc_curve, auc, RocCurveDisplay
from sklearn.preprocessing import LabelBinarizer

In [3]:
import nltk

# Download necessary NLTK data for VADER and POS tagging
# This might take a moment if not already downloaded
try:
    nltk.data.find('sentiment/vader_lexicon.zip')
except LookupError:
    nltk.download('vader_lexicon')
try:
    nltk.data.find('taggers/averaged_perceptron_tagger.zip')
except LookupError:
    nltk.download('averaged_perceptron_tagger')
try:
    nltk.data.find('taggers/averaged_perceptron_tagger_eng.zip')
except LookupError:
    nltk.download('averaged_perceptron_tagger_eng')

# Download required NLTK data (run once)
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [4]:
# load extended data for text sentiment
df = pd.read_parquet('datasets/with_stop_word/amazon_user_reviews_textANDfeature_sentiment_with_sw.parquet')

df.head()

,original_text,sentiment,exclamation_count,question_count,word_count,char_count,all_caps_words,uppercase_ratio,total_punctuation,avg_word_length,helpful_vote,int_verified_purchase,hour,month,season
0,The material isn’t what I expected. I thought ...,2,0,0,14,73,0,0.041096,2,5.214286,0,1,20,8,2
1,the magnetic snap tore out of the fake leather...,2,0,0,14,70,0,0.000000,1,5.000000,0,0,17,9,3
2,[[VIDEOID:f43161a0116bd3fb943546242e053d92]] T...,2,1,0,44,293,1,0.044369,5,6.659091,0,0,3,11,3
3,Would not recommend,2,0,0,3,19,0,0.052632,0,6.333333,0,1,1,10,3
4,"they are all too small, but very pretty",2,0,0,8,39,0,0.000000,1,4.875000,0,1,17,1,4


## Further Feature Development

In [5]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer

# Initialize VADER sentiment intensity analyzer
sia = SentimentIntensityAnalyzer()

# Function to get VADER sentiment scores
def get_vader_sentiment(text):
    vs = sia.polarity_scores(text)
    return vs['neg'], vs['neu'], vs['pos'], vs['compound']

# Apply the function to the original text data to generate features for the entire dataframe
df_vader_features = df['original_text'].apply(lambda x: pd.Series(get_vader_sentiment(x), index=['vader_neg', 'vader_neu', 'vader_pos', 'vader_compound']))

# Concatenate VADER features to the main dataframe
df = pd.concat([df, df_vader_features], axis=1)

print("VADER Sentiment Features added to DataFrame (first 5 rows with new features):")
display(df[['original_text', 'vader_neg', 'vader_neu', 'vader_pos', 'vader_compound']].head())

VADER Sentiment Features added to DataFrame (first 5 rows with new features):


,original_text,vader_neg,vader_neu,vader_pos,vader_compound
0,The material isn’t what I expected. I thought ...,0.000,1.000,0.000,0.0000
1,the magnetic snap tore out of the fake leather...,0.205,0.795,0.000,-0.4767
2,[[VIDEOID:f43161a0116bd3fb943546242e053d92]] T...,0.000,0.798,0.202,0.8718
3,Would not recommend,0.513,0.487,0.000,-0.2755
4,"they are all too small, but very pretty",0.000,0.596,0.404,0.6946


In [6]:
from nltk.tokenize import word_tokenize

# Function to get POS tag counts
def get_pos_features(text):
    tokens = word_tokenize(text)
    tagged = nltk.pos_tag(tokens)
    pos_counts = {
        'NN_count': 0, 'JJ_count': 0, 'VB_count': 0, 'RB_count': 0, # Nouns, Adjectives, Verbs, Adverbs
        'DT_count': 0, 'PRP_count': 0, # Determiners, Pronouns
        'PUNCT_count': 0 # Punctuation
    }
    for word, tag in tagged:
        if tag.startswith('NN'): pos_counts['NN_count'] += 1
        elif tag.startswith('JJ'): pos_counts['JJ_count'] += 1
        elif tag.startswith('VB'): pos_counts['VB_count'] += 1
        elif tag.startswith('RB'): pos_counts['RB_count'] += 1
        elif tag.startswith('DT'): pos_counts['DT_count'] += 1
        elif tag.startswith('PRP'): pos_counts['PRP_count'] += 1
        elif tag in ['.', ',', '!', '?']: pos_counts['PUNCT_count'] += 1
    return pd.Series(pos_counts)

# Apply the function to the original text data to generate features for the entire dataframe
df_pos_features = df['original_text'].apply(get_pos_features)

# Concatenate POS features to the main dataframe
df = pd.concat([df, df_pos_features], axis=1)

print("POS Tagging Features added to DataFrame (first 5 rows with new features):")
display(df[['original_text', 'NN_count', 'JJ_count', 'VB_count', 'RB_count', 'DT_count', 'PRP_count', 'PUNCT_count']].head())

POS Tagging Features added to DataFrame (first 5 rows with new features):


,original_text,NN_count,JJ_count,VB_count,RB_count,DT_count,PRP_count,PUNCT_count
0,The material isn’t what I expected. I thought ...,5,0,4,0,2,3,2
1,the magnetic snap tore out of the fake leather...,5,2,1,0,3,0,1
2,[[VIDEOID:f43161a0116bd3fb943546242e053d92]] T...,26,4,10,5,5,3,5
3,Would not recommend,0,0,1,1,0,0,0
4,"they are all too small, but very pretty",0,1,1,3,1,1,1


In [7]:
import pandas as pd
from scipy.sparse import hstack

# Drop rows where 'original_text' is NaN or empty string
df.dropna(subset=['original_text'], inplace=True)
df = df[df['original_text'].str.strip() != '']

# Split data into features (X) and target (y)
X_all_features = df.drop('sentiment', axis=1)
y = df['sentiment']

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_all_features, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training data size: {X_train.shape[0]} samples")
print(f"Testing data size: {X_test.shape[0]} samples")

# Initialize TF-IDF Vectorizer
tfidf_vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))

# Fit and transform the 'original_text' column for training and testing data
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train['original_text'])
X_test_tfidf = tfidf_vectorizer.transform(X_test['original_text'])

# Drop the 'original_text' column from X_train and X_test to concatenate with TF-IDF features
X_train_numerical = X_train.drop(columns=['original_text'])
X_test_numerical = X_test.drop(columns=['original_text'])

# Convert numerical features to sparse matrix format if they are not already (hstack requires sparse or numpy arrays)
X_train_numerical_sparse = X_train_numerical.sparse.to_coo() if pd.api.types.is_sparse(X_train_numerical) else X_train_numerical.values
X_test_numerical_sparse = X_test_numerical.sparse.to_coo() if pd.api.types.is_sparse(X_test_numerical) else X_test_numerical.values

# Combine TF-IDF features with other numerical features
X_train_combined = hstack([X_train_tfidf, X_train_numerical_sparse])
X_test_combined = hstack([X_test_tfidf, X_test_numerical_sparse])

print(f"TF-IDF vocabulary size: {len(tfidf_vectorizer.vocabulary_)}")
print(f"Shape of X_train_combined: {X_train_combined.shape}")
print(f"Shape of X_test_combined: {X_test_combined.shape}")

Training data size: 14720 samples
Testing data size: 3680 samples
TF-IDF vocabulary size: 5000
Shape of X_train_combined: (14720, 5024)
Shape of X_test_combined: (3680, 5024)


/tmp/ipykernel_3320/4171190665.py:30: DeprecationWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  X_train_numerical_sparse = X_train_numerical.sparse.to_coo() if pd.api.types.is_sparse(X_train_numerical) else X_train_numerical.values
/tmp/ipykernel_3320/4171190665.py:31: DeprecationWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  X_test_numerical_sparse = X_test_numerical.sparse.to_coo() if pd.api.types.is_sparse(X_test_numerical) else X_test_numerical.values


### Model Training and Evaluation

#### Logistic Regression

In [8]:
# Initialize and train Logistic Regression model
lr_model = LogisticRegression(max_iter=1000, solver='liblinear', random_state=42)
lr_model.fit(X_train_combined, y_train)

# Predict on the test set
y_pred_lr = lr_model.predict(X_test_combined)

# Evaluate the model
print("Logistic Regression Performance:")
print(f"Accuracy: {accuracy_score(y_test, y_pred_lr):.4f}")
print(classification_report(y_test, y_pred_lr))

Logistic Regression Performance:
Accuracy: 0.7084
              precision    recall  f1-score   support

           0       0.78      0.81      0.80      1236
           1       0.64      0.57      0.60      1217
           2       0.69      0.74      0.71      1227

    accuracy                           0.71      3680
   macro avg       0.70      0.71      0.71      3680
weighted avg       0.71      0.71      0.71      3680



#### Multinomial Naive Bayes


In [9]:
# Initialize and train Multinomial Naive Bayes model
mnb_model = MultinomialNB()
mnb_model.fit(X_train_combined, y_train)

# Predict on the test set
y_pred_mnb = mnb_model.predict(X_test_combined)

# Evaluate the model
print("Multinomial Naive Bayes Performance:")
print(f"Accuracy: {accuracy_score(y_test, y_pred_mnb):.4f}")
print(classification_report(y_test, y_pred_mnb))

ValueError: Negative values in data passed to MultinomialNB (input X).

#### Support Vector Machine (SVM)

In [ ]:
# Initialize and train SVM model
# Using a linear kernel often works well for text data due to high dimensionality
svm_model = SVC(kernel='linear', random_state=42)
svm_model.fit(X_train_combined, y_train)

# Predict on the test set
y_pred_svm = svm_model.predict(X_test_combined)

# Evaluate the model
print("Support Vector Machine (SVM) Performance:")
print(f"Accuracy: {accuracy_score(y_test, y_pred_svm):.4f}")
print(classification_report(y_test, y_pred_svm))

#### Random Forest

In [ ]:
# Initialize and train Random Forest model
# n_estimators is the number of trees in the forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train_combined, y_train)

# Predict on the test set
y_pred_rf = rf_model.predict(X_test_combined)

# Evaluate the model
print("Random Forest Performance:")
print(f"Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}")
print(classification_report(y_test, y_pred_rf))

### Confusion Matrices visualization

In [ ]:
models = {
    "Logistic Regression": (lr_model, y_pred_lr),
    "Multinomial Naive Bayes": (mnb_model, y_pred_mnb),
    "SVM": (svm_model, y_pred_svm),
    "Random Forest": (rf_model, y_pred_rf)
}

for name, (model, y_pred) in models.items():
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Negative', 'Neutral', 'Positive'],
                yticklabels=['Negative', 'Neutral', 'Positive'])
    plt.title(f'Confusion Matrix for {name}')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.show()

### ROC Curves Visualization

In [ ]:
lb = LabelBinarizer()
# y_test_binarized will have columns for each class (0, 1, 2)
y_test_binarized = lb.fit_transform(y_test)


plt.figure(figsize=(15, 10))

# --- Logistic Regression ROC ---
plt.subplot(2, 2, 1)
y_score_lr = lr_model.predict_proba(X_test_combined)
RocCurveDisplay.from_predictions(y_test_binarized[:, 0], y_score_lr[:, 0], name=f'Class 0 (AUC={auc(roc_curve(y_test_binarized[:, 0], y_score_lr[:, 0])[0], roc_curve(y_test_binarized[:, 0], y_score_lr[:, 0])[1]):.2f})', pos_label=1, ax=plt.gca())
RocCurveDisplay.from_predictions(y_test_binarized[:, 1], y_score_lr[:, 1], name=f'Class 1 (AUC={auc(roc_curve(y_test_binarized[:, 1], y_score_lr[:, 1])[0], roc_curve(y_test_binarized[:, 1], y_score_lr[:, 1])[1]):.2f})', pos_label=1, ax=plt.gca())
RocCurveDisplay.from_predictions(y_test_binarized[:, 2], y_score_lr[:, 2], name=f'Class 2 (AUC={auc(roc_curve(y_test_binarized[:, 2], y_score_lr[:, 2])[0], roc_curve(y_test_binarized[:, 2], y_score_lr[:, 2])[1]):.2f})', pos_label=1, ax=plt.gca())
plt.title('ROC Curve for Logistic Regression')
plt.plot([0, 1], [0, 1], 'k--', label='Chance Level')
plt.legend()


# --- Multinomial Naive Bayes ROC ---
plt.subplot(2, 2, 2)
y_score_mnb = mnb_model.predict_proba(X_test_combined)
RocCurveDisplay.from_predictions(y_test_binarized[:, 0], y_score_mnb[:, 0], name=f'Class 0 (AUC={auc(roc_curve(y_test_binarized[:, 0], y_score_mnb[:, 0])[0], roc_curve(y_test_binarized[:, 0], y_score_mnb[:, 0])[1]):.2f})', pos_label=1, ax=plt.gca())
RocCurveDisplay.from_predictions(y_test_binarized[:, 1], y_score_mnb[:, 1], name=f'Class 1 (AUC={auc(roc_curve(y_test_binarized[:, 1], y_score_mnb[:, 1])[0], roc_curve(y_test_binarized[:, 1], y_score_mnb[:, 1])[1]):.2f})', pos_label=1, ax=plt.gca())
RocCurveDisplay.from_predictions(y_test_binarized[:, 2], y_score_mnb[:, 2], name=f'Class 2 (AUC={auc(roc_curve(y_test_binarized[:, 2], y_score_mnb[:, 2])[0], roc_curve(y_test_binarized[:, 2], y_score_mnb[:, 2])[1]):.2f})', pos_label=1, ax=plt.gca())
plt.title('ROC Curve for Multinomial Naive Bayes')
plt.plot([0, 1], [0, 1], 'k--', label='Chance Level')
plt.legend()


# --- SVM (SVC) ROC (Requires probability=True during SVC initialization to get predict_proba) ---
# As SVM was initialized without probability=True, we can't directly use predict_proba.
# If you need ROC for SVM, you would need to re-initialize and retrain it with probability=True.
# For now, we will skip SVM ROC or provide a placeholder.
# If probability=True was set, the code would look like this:
# plt.subplot(2, 2, 3)
# y_score_svm = svm_model.predict_proba(X_test_tfidf)
# RocCurveDisplay.from_predictions(y_test_binarized[:, 0], y_score_svm[:, 0], name=f'Class 0 (AUC={auc(roc_curve(y_test_binarized[:, 0], y_score_svm[:, 0])[0], roc_curve(y_test_binarized[:, 0], y_score_svm[:, 0])[1]):.2f})', pos_label=1, ax=plt.gca())
# ... and so on for other classes
# plt.title('ROC Curve for SVM')
# plt.plot([0, 1], [0, 1], 'k--', label='Chance Level')
# plt.legend()


# --- Random Forest ROC ---
plt.subplot(2, 2, 4)
y_score_rf = rf_model.predict_proba(X_test_combined)
RocCurveDisplay.from_predictions(y_test_binarized[:, 0], y_score_rf[:, 0], name=f'Class 0 (AUC={auc(roc_curve(y_test_binarized[:, 0], y_score_rf[:, 0])[0], roc_curve(y_test_binarized[:, 0], y_score_rf[:, 0])[1]):.2f})', pos_label=1, ax=plt.gca())
RocCurveDisplay.from_predictions(y_test_binarized[:, 1], y_score_rf[:, 1], name=f'Class 1 (AUC={auc(roc_curve(y_test_binarized[:, 1], y_score_rf[:, 1])[0], roc_curve(y_test_binarized[:, 1], y_score_rf[:, 1])[1]):.2f})', pos_label=1, ax=plt.gca())
RocCurveDisplay.from_predictions(y_test_binarized[:, 2], y_score_rf[:, 2], name=f'Class 2 (AUC={auc(roc_curve(y_test_binarized[:, 2], y_score_rf[:, 2])[0], roc_curve(y_test_binarized[:, 2], y_score_rf[:, 2])[1]):.2f})', pos_label=1, ax=plt.gca())
plt.title('ROC Curve for Random Forest')
plt.plot([0, 1], [0, 1], 'k--', label='Chance Level')
plt.legend()

plt.tight_layout()
plt.show()